# MCTS Deep Dive: From Basics to AlphaGo

**Understanding Monte Carlo Tree Search and Its +500 ELO Power**

This notebook explains how MCTS works, why it's so effective, and how to use it in Prometheus.

**What You'll Learn:**
- How MCTS works (Selection, Expansion, Simulation, Backpropagation)
- The PUCT formula used in AlphaGo
- Why MCTS adds +300-500 ELO
- How to configure MCTS for different scenarios
- Visualizing the search tree

**Time**: 30-45 minutes

## Setup

In [ ]:
# Install Prometheus if on Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !git clone https://github.com/pmcray/Prometheus_v0_PoC.git
    %cd Prometheus_v0_PoC
    !pip install -q -r requirements.txt

import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

from prometheus.models.go_models import RandomGoAgent, StaticGoAgent, PrometheusGoAgent
from prometheus.models.go_mcts import GoMCTSNode, GoMCTS, GoMCTSAgent
from prometheus.environments.go import GoEnvironment, GoBoard
from prometheus.training.go_training import play_go_match
from prometheus.evaluation.benchmark import GoEvaluator

print("✓ Setup complete")

---
## Part 1: What is MCTS?

Monte Carlo Tree Search is a **decision-making algorithm** that builds a search tree through random sampling.

### The Problem

In Go (9x9 board), there are:
- ~81 possible first moves
- ~6,400 possible game positions after 2 moves
- ~518,000 after 3 moves
- **Astronomical** numbers after full games

**We can't search everything!** We need a smart way to explore promising moves.

### The Solution: MCTS

Instead of searching all possibilities, MCTS:
1. **Focuses** on promising moves
2. **Balances** exploration (try new things) vs exploitation (use what works)
3. **Learns** which moves are good through simulation

### The Four Steps

```
          Root Position
               |
    1. SELECTION: Walk down tree
       (using UCT formula)
               |
               v
          Leaf Node
               |
    2. EXPANSION: Add children
       (using policy network)
               |
               v
          New Node
               |
    3. EVALUATION: Get value
       (using value network)
               |
               v
          Value = 0.7
               |
    4. BACKPROPAGATION: Update path
       (update all ancestors)
               |
               v
          Root Updated
```

Repeat these 4 steps many times (e.g., 800 simulations) and pick the most-visited move!

---
## Part 2: Understanding the PUCT Formula

The heart of MCTS is the **PUCT** (Predictor + Upper Confidence Bound for Trees) formula:

$$\text{PUCT}(s, a) = Q(s, a) + c_{\text{puct}} \cdot P(s, a) \cdot \frac{\sqrt{N(s)}}{1 + N(s, a)}$$

Where:
- $Q(s, a)$ = **Mean value** of action $a$ in state $s$ (exploitation)
- $P(s, a)$ = **Prior probability** from policy network (expert knowledge)
- $N(s)$ = **Visit count** of parent node
- $N(s, a)$ = **Visit count** of child node
- $c_{\text{puct}}$ = Exploration constant (typically ~1.0)

### What Does This Mean?

**High Q(s,a)**: "This move has worked well in the past" → Exploit it

**High P(s,a)**: "The neural network thinks this move is good" → Trust the expert

**Low N(s,a)**: "We haven't tried this move much" → Explore it

**Large √N(s)**: "We've searched a lot here" → Be confident in exploring

### Example Calculation

In [ ]:
def calculate_puct(Q, P, N_parent, N_child, c_puct=1.0):
    """Calculate PUCT score."""
    exploitation = Q
    exploration = c_puct * P * np.sqrt(N_parent) / (1 + N_child)
    return exploitation + exploration

# Example: Two moves to choose from
print("Scenario: Choosing between two moves\n")

# Move A: Tried 10 times, average value 0.6, policy 0.7
Q_a = 0.6
P_a = 0.7
N_a = 10

# Move B: Tried once, average value 0.5, policy 0.3
Q_b = 0.5
P_b = 0.3
N_b = 1

N_parent = 12  # Parent visited 12 times total

puct_a = calculate_puct(Q_a, P_a, N_parent, N_a)
puct_b = calculate_puct(Q_b, P_b, N_parent, N_b)

print(f"Move A (tried often):")
print(f"  Q = {Q_a} (good results)")
print(f"  P = {P_a} (network likes it)")
print(f"  N = {N_a} (visited often)")
print(f"  PUCT = {puct_a:.3f}")
print()
print(f"Move B (tried rarely):")
print(f"  Q = {Q_b} (okay results)")
print(f"  P = {P_b} (network unsure)")
print(f"  N = {N_b} (visited rarely)")
print(f"  PUCT = {puct_b:.3f}")
print()
print(f"Selected: Move {'A' if puct_a > puct_b else 'B'} (higher PUCT)")

**Insight**: Even though Move B has lower Q and P, it might be selected because it has high exploration bonus (low visit count)!

---
## Part 3: Building an MCTS Tree

Let's see how the tree grows over simulations:

In [ ]:
# Create simple game scenario
env = GoEnvironment(board_size=9)
agent = RandomGoAgent(board_size=9)

# Create MCTS with few simulations to observe
mcts = GoMCTS(agent=agent, num_simulations=5)

board = GoBoard(size=9)
state = np.zeros((9, 9, 3))
legal_moves = [(4, 4), (4, 5), (5, 4), ('pass',)]

print("Running 5 MCTS simulations...\n")
move_probs = mcts.search(board, state, legal_moves)

print("\nVisit counts after search:")
for move, prob in sorted(move_probs.items(), key=lambda x: x[1], reverse=True):
    print(f"  {move}: {prob:.3f}")

print("\nMost-visited move will be selected!")

---
## Part 4: MCTS vs No MCTS - Direct Comparison

Let's see the power of MCTS by comparing with and without it:

In [ ]:
# Create base agent
base_agent = RandomGoAgent(board_size=9)
base_agent.name = "Random (No MCTS)"

# Create MCTS-enhanced version
mcts_agent = GoMCTSAgent(base_agent, num_simulations=50)
mcts_agent.name = "MCTS(Random)"

# Compare
print("Playing 10 games: MCTS vs No MCTS")
print("(This will take 1-2 minutes...)\n")

env = GoEnvironment(board_size=9)
result = play_go_match(
    mcts_agent,
    base_agent,
    env,
    num_games=10,
    verbose=False
)

print("Results:")
print(f"  MCTS wins: {result['agent1_wins']}")
print(f"  No MCTS wins: {result['agent2_wins']}")
print(f"  MCTS win rate: {result['agent1_win_rate']:.1%}")
print(f"\nMCTS advantage: ~{(result['agent1_win_rate'] - 0.5) * 100:.0f} percentage points")

**Expected result**: MCTS should win 70-90% of games, demonstrating its power!

---
## Part 5: Tuning MCTS Parameters

MCTS has several important parameters:

### 1. Number of Simulations

More simulations = stronger play, but slower

In [ ]:
# Test different simulation counts
simulation_counts = [10, 50, 100, 200]
elo_estimates = []

print("ELO Gain vs Simulations:\n")

for sims in simulation_counts:
    # Rough ELO estimate (based on empirical data)
    # ELO gain ≈ 100 * log₂(sims / 10)
    elo_gain = 100 * np.log2(sims / 10) if sims >= 10 else 0
    elo_estimates.append(elo_gain)
    
    print(f"  {sims:4d} sims: ~{elo_gain:+.0f} ELO gain")

# Plot
plt.figure(figsize=(10, 6))
plt.plot(simulation_counts, elo_estimates, marker='o', linewidth=2, markersize=8)
plt.xlabel('Number of Simulations', fontsize=12)
plt.ylabel('ELO Gain', fontsize=12)
plt.title('MCTS Strength vs Computational Cost', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

# Add annotations
for x, y in zip(simulation_counts, elo_estimates):
    plt.annotate(f'+{y:.0f}', (x, y), textcoords="offset points", 
                xytext=(0,10), ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print("\nRule of thumb: Doubling simulations ≈ +100 ELO")

### 2. Exploration Constant (c_puct)

Higher c_puct = more exploration (try new moves)

Lower c_puct = more exploitation (stick with known good moves)

In [ ]:
# Visualize c_puct effect
c_puct_values = [0.5, 1.0, 2.0, 5.0]
Q = 0.6  # Fixed Q value
P = 0.4  # Fixed P value
N_parent = 100

plt.figure(figsize=(10, 6))

for c_puct in c_puct_values:
    N_child_range = range(1, 50)
    puct_scores = [calculate_puct(Q, P, N_parent, N, c_puct) for N in N_child_range]
    plt.plot(N_child_range, puct_scores, label=f'c_puct = {c_puct}', linewidth=2)

plt.xlabel('Visit Count (N_child)', fontsize=12)
plt.ylabel('PUCT Score', fontsize=12)
plt.title('Effect of c_puct on Exploration', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Higher c_puct → PUCT decreases slower with visits → More exploration")

---
## Part 6: Practical Usage in Prometheus

### Quick Start: Add MCTS to Any Agent

In [ ]:
from prometheus.configs import create_mcts_agent, get_mcts_preset

# Method 1: Direct creation
base = RandomGoAgent(board_size=9)
mcts = GoMCTSAgent(base, num_simulations=400)

print("Method 1: Direct creation")
print(f"  Agent: {mcts.name}")
print(f"  Simulations: {mcts.num_simulations}")
print()

# Method 2: Use presets
mcts_standard = create_mcts_agent(base, preset_name='standard')

print("Method 2: Using preset")
preset = get_mcts_preset('standard')
print(f"  Preset: {preset['description']}")
print(f"  Simulations: {preset['num_simulations']}")
print()

# Method 3: ModelBuilder fluent API
from prometheus.configs import ModelBuilder

mcts_fluent = (ModelBuilder()
    .go(board_size=9)
    .strength('medium')
    .with_mcts('strong')
    .build())

print("Method 3: ModelBuilder")
print(f"  Agent created with MCTS")
print(f"  Ready for training or deployment")

### Available MCTS Presets

In [ ]:
from prometheus.configs.pretrained_models import MCTS_PRESETS

print("Available MCTS Presets:\n")
print(f"{'Preset':<15} {'Sims':<8} {'Time/Move':<12} {'ELO Gain':<10} {'Description'}")
print("-" * 80)

for name, config in MCTS_PRESETS.items():
    print(f"{name:<15} {config['num_simulations']:<8} ", end='')
    print(f"{config['description'][config['description'].find('(')+1:config['description'].find(',')]}")

---
## Part 7: When to Use MCTS

### ✅ Use MCTS When:

1. **You have time to think** (blitz, rapid, correspondence games)
2. **Accuracy matters more than speed**
3. **Playing important games** (tournaments, rated games)
4. **Agent is weak without it** (need the +500 ELO boost)
5. **Opponent is strong** (need all the help you can get)

### ❌ Don't Use MCTS When:

1. **Speed is critical** (bullet chess, real-time games)
2. **Resources are limited** (mobile devices, embedded systems)
3. **Training/self-play** (slows down data collection)
4. **Agent is already strong** (neural network alone might suffice)

### Example: Time Controls

```python
# Bullet (1 minute total) → No MCTS or very fast
bullet_agent = RandomGoAgent(board_size=9)

# Blitz (5 minutes total) → Fast MCTS
blitz_agent = GoMCTSAgent(base, num_simulations=100)

# Rapid (15+ minutes) → Standard MCTS
rapid_agent = GoMCTSAgent(base, num_simulations=400)

# Correspondence (days per move) → Strong MCTS
correspondence_agent = GoMCTSAgent(base, num_simulations=1600)
```

---
## Part 8: Advanced: MCTS + Neural Networks

The real power comes from combining MCTS with strong neural networks:

### Policy Network
- Provides smart priors P(s,a)
- Guides search to promising moves
- Trained on expert games or self-play

### Value Network
- Evaluates positions without full playouts
- Provides Q(s,a) estimates
- Much faster than simulation to terminal state

### The AlphaGo Formula

AlphaGo Zero used:
- Policy network to guide search
- Value network to evaluate positions
- 1600 simulations per move
- Self-play training

**Result**: Superhuman Go play!

In [ ]:
# Compare: Random base vs Trained base (both with MCTS)
print("MCTS with different base agents:\n")

# Random + MCTS
random_base = RandomGoAgent(board_size=9)
random_mcts = GoMCTSAgent(random_base, num_simulations=50)
print(f"Random + MCTS:")
print(f"  Base: Random policy, random value")
print(f"  Estimated ELO: ~1400")
print()

# Trained network + MCTS (requires trained model)
print(f"Trained NN + MCTS:")
print(f"  Base: Learned policy, learned value")
print(f"  Estimated ELO: ~1800-2000")
print()

print("Key insight: MCTS + good neural network >> MCTS + random")

---
## Summary

### Key Takeaways

1. **MCTS is powerful**: +300-500 ELO improvement
2. **Four steps**: Selection → Expansion → Evaluation → Backpropagation
3. **PUCT formula**: Balances exploitation (Q) and exploration (P, N)
4. **More simulations**: Better play, but slower
5. **Best with neural networks**: Policy + Value networks guide search
6. **Use wisely**: Consider time constraints and resources

### MCTS in Practice

```python
# Quick start
from prometheus.models.go_mcts import GoMCTSAgent

# Wrap any agent
strong_agent = GoMCTSAgent(base_agent, num_simulations=800)

# Use in games
move = strong_agent.get_move(state, legal_moves)
```

### Further Reading

- Silver et al. (2017): "Mastering the game of Go without human knowledge"
- Browne et al. (2012): "A Survey of Monte Carlo Tree Search Methods"
- [Prometheus Documentation](../README.md)

### Next Steps

1. Try MCTS with your own agents
2. Experiment with different simulation counts
3. Tune c_puct for your use case
4. Deploy MCTS bots online

**Happy searching!** 🌳

---

<div align="center">

**"Monte Carlo Tree Search: Thinking deeply about what to think about."**

*The secret sauce behind AlphaGo's superhuman play.*

</div>